In [26]:
import pandas as pd
import numpy as np

In [27]:
df = pd.read_csv('../data/raw/sessions.csv')

In [28]:
df.shape

(5000, 22)

In [29]:
df.isnull().sum()

session_id                       0
user_id                          0
session_start                    0
session_date                     0
source                           0
device                           0
region                           0
age_band                         0
income_band                      0
product                          0
is_returning_user                0
page_views                       0
session_duration_seconds         0
completed_stage                  0
product_page_viewed              0
calculator_used                  0
cta_clicked                      0
form_started                     0
form_submitted                   0
exit_page                      779
exit_reason                    779
estimated_opportunity_value      0
dtype: int64

In [30]:
df['session_id'].duplicated().sum()

np.int64(0)

In [31]:
df.dtypes

session_id                       str
user_id                          str
session_start                    str
session_date                     str
source                           str
device                           str
region                           str
age_band                         str
income_band                      str
product                          str
is_returning_user              int64
page_views                     int64
session_duration_seconds       int64
completed_stage                  str
product_page_viewed            int64
calculator_used                int64
cta_clicked                    int64
form_started                   int64
form_submitted                 int64
exit_page                        str
exit_reason                      str
estimated_opportunity_value    int64
dtype: object

In [32]:
df['session_date'] = pd.to_datetime(df['session_date'])

In [33]:
df['session_start'] = pd.to_datetime(df['session_start'])

## Creating session month and week number columns

In [34]:
df['session_month'] = df['session_date'].dt.month

In [35]:
df['session_week'] = df['session_date'].dt.isocalendar().week.astype(int)

In [36]:
df.dtypes

session_id                                str
user_id                                   str
session_start                  datetime64[us]
session_date                   datetime64[us]
source                                    str
device                                    str
region                                    str
age_band                                  str
income_band                               str
product                                   str
is_returning_user                       int64
page_views                              int64
session_duration_seconds                int64
completed_stage                           str
product_page_viewed                     int64
calculator_used                         int64
cta_clicked                             int64
form_started                            int64
form_submitted                          int64
exit_page                                 str
exit_reason                               str
estimated_opportunity_value       

In [37]:
df.head()

,session_id,user_id,session_start,session_date,source,device,region,age_band,income_band,product,...,product_page_viewed,calculator_used,cta_clicked,form_started,form_submitted,exit_page,exit_reason,estimated_opportunity_value,session_month,session_week
0,S000001,U000179,2025-05-20 15:42:34,2025-05-20,Email,Desktop,North,45-54,50K-100K,Insurance Plan,...,1,0,0,0,0,Product Page Viewed,High interest-rate concern,5699,5,21
1,S000002,U001109,2025-06-09 01:31:53,2025-06-09,Email,Desktop,West,55+,100K+,Credit Card,...,1,1,1,1,1,NaN,NaN,0,6,24
2,S000003,U000741,2025-01-14 11:16:09,2025-01-14,Organic Search,Mobile,East,35-44,25K-50K,Personal Loan,...,1,0,0,0,0,Product Page Viewed,Unclear product value,76835,1,3
3,S000004,U001665,2025-02-05 19:19:03,2025-02-05,Email,Desktop,West,25-34,50K-100K,Savings Account,...,1,1,1,1,1,NaN,NaN,0,2,6
4,S000005,U001590,2025-02-24 14:28:05,2025-02-24,Paid Ads,Mobile,South,25-34,Below 25K,Insurance Plan,...,1,0,0,0,0,Product Page Viewed,High interest-rate concern,9572,2,9


### Convertin columns into category dtype

In [15]:
cols = ['source', 'device', 'region', 'product']
for cols in cols:
    df[cols] = df[cols].astype('category')
print(f"Converted Success")

Converted Success


In [16]:
df.dtypes

session_id                                str
user_id                                   str
session_start                  datetime64[us]
session_date                   datetime64[us]
source                               category
device                               category
region                               category
age_band                                  str
income_band                               str
product                              category
is_returning_user                       int64
page_views                              int64
session_duration_seconds                int64
completed_stage                           str
product_page_viewed                     int64
calculator_used                         int64
cta_clicked                             int64
form_started                            int64
form_submitted                          int64
exit_page                                 str
exit_reason                               str
estimated_opportunity_value       

In [21]:
funnel_columns = [
    'product_page_viewed',
    'calculator_used',
    'cta_clicked',
    'form_started',
    'form_submitted'
]

for column in funnel_columns:
    invalid_values = ~df[column].isin([0, 1])
    print(column, 'invalid values:', invalid_values.sum())

funnel_order_checks = {
    'calculator_without_product_page': ((df['calculator_used'] == 1) & (df['product_page_viewed'] == 0)).sum(),
    'cta_without_calculator': ((df['cta_clicked'] == 1) & (df['calculator_used'] == 0)).sum(),
    'form_without_cta': ((df['form_started'] == 1) & (df['cta_clicked'] == 0)).sum(),
    'submit_without_form': ((df['form_submitted'] == 1) & (df['form_started'] == 0)).sum(),
}

print(funnel_order_checks)


product_page_viewed invalid values: 0
calculator_used invalid values: 0
cta_clicked invalid values: 0
form_started invalid values: 0
form_submitted invalid values: 0
{'calculator_without_product_page': np.int64(0), 'cta_without_calculator': np.int64(0), 'form_without_cta': np.int64(0), 'submit_without_form': np.int64(0)}


In [22]:
print('Negative page views:', (df['page_views'] < 0).sum())
print('Negative session duration:', (df['session_duration_seconds'] < 0).sum())
print('Negative opportunity value:', (df['estimated_opportunity_value'] < 0).sum())
print('Invalid returning-user flags:', (~df['is_returning_user'].isin([0, 1])).sum())


Negative page views: 0
Negative session duration: 0
Negative opportunity value: 0
Invalid returning-user flags: 0


In [23]:
df['source'].unique()

['Email', 'Organic Search', 'Paid Ads', 'Direct', 'Social Media', 'Referral']
Categories (6, str): ['Direct', 'Email', 'Organic Search', 'Paid Ads', 'Referral', 'Social Media']

In [24]:
df.shape

(5000, 24)

In [25]:
df.to_csv('../data/cleaned/cleaned_sessions.csv')